# 🏀 Fantasy NBA Draft Engine — 2026-27
**Budget:** $200 × 10 teams | **Roster:** 10 starters + 3 bench | **Positions:** PG, SG, SF, PF, C, G, F, F2, G2, FLEX

### Strategy: Punt 4 — Dominate 6
Target **PTS + AST + STL + PTS_MIN + FG3M + FT%** (correlated guard/wing stats).  
Concede REB, BLK, DD, FG% — lock 6 categories every matchup → guaranteed 6-4 win floor.

---
**Cells:**  
1 · Imports  
2 · Game logs (2025-26 + 2024-25 only)  
3 · Active-player metadata: Age, Position, Height  
4 · Clean & Feature Engineering  
5 · Aggregate Stats (rookie-aware)  
6 · Reliability + Age Factors  
7 · Predictions & Master Merge  
8 · Z-Scores — Full 10-cat + 6-cat Punt  
9 · Auction Values (dual: all-cats & 6-cat)  
10 · Optimal Team Builder (6-cat punt mode)  
11 · 10-Team League Simulation  
12 · Category Win Projection  
13 · Save Excel

In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ═══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from nba_api.stats.endpoints import (
    leaguegamelog,
    commonallplayers,
    commonplayerinfo,
)

print('✅ Imports OK')

✅ Imports OK


In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — GAME LOGS: 2024-25 + 2025-26 ONLY
# ═══════════════════════════════════════════════════════════════════
# We only need 2 seasons for prediction (70/30 weight).
# For reliability we use GP within these 2 seasons rather than 10-year history.
# Rationale: recent form is more predictive than decade-old data;
#            shorter fetch = faster, fewer rate-limit bans.

SEASONS = ['2024-25', '2025-26']
CURRENT_SEASON = '2025-26'
PREV_SEASON    = '2024-25'
MAX_GAMES      = 82
DRAFT_YEAR_CURRENT = 2025   # players drafted 2025 → 2025-26 rookies
DRAFT_DATE     = pd.Timestamp('2026-10-01')

all_seasons_data = []
print('Fetching 2 seasons of game logs (~10s)...')

for season in SEASONS:
    print(f'  → {season}', end='', flush=True)
    try:
        gl = leaguegamelog.LeagueGameLog(
            season=season,
            player_or_team_abbreviation='P',
            season_type_all_star='Regular Season'
        )
        df_s = gl.get_data_frames()[0]
        df_s['SEASON'] = season
        all_seasons_data.append(df_s)
        print(f' [{len(df_s):,} rows]')
        time.sleep(2.5)
    except Exception as e:
        print(f' ERROR: {e}')

df_all_stats = pd.concat(all_seasons_data, ignore_index=True)
print(f'\n✅ Total rows: {len(df_all_stats):,}')
print('Columns:', list(df_all_stats.columns))

Fetching 2 seasons of game logs (~10s)...
  → 2024-25 [26,306 rows]
  → 2025-26 [23,927 rows]

✅ Total rows: 50,233
Columns: ['SEASON_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'FANTASY_PTS', 'VIDEO_AVAILABLE', 'SEASON']


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — ACTIVE-PLAYER METADATA: Age · Position · Height
# ═══════════════════════════════════════════════════════════════════
# Strategy: only fetch commonplayerinfo for players who appeared in
# the 2025-26 season (CURRENT_SEASON). This is ~350-450 players
# instead of every player since 2015, making this ~5× faster and
# avoiding unnecessary lookups for retired players.
#
# commonplayerinfo returns per-player:
#   BIRTHDATE, POSITION, HEIGHT (feet-inches string), DRAFT_YEAR

active_ids = (
    df_all_stats[df_all_stats['SEASON'] == CURRENT_SEASON]['PLAYER_ID']
    .unique()
)
print(f'Fetching metadata for {len(active_ids)} active players...')
print('Estimated time: ~{:.0f} min'.format(len(active_ids) * 0.65 / 60))

player_meta = []
errors_meta = []

for i, pid in enumerate(active_ids):
    try:
        info = commonplayerinfo.CommonPlayerInfo(player_id=pid)
        row  = info.get_data_frames()[0].iloc[0]

        # ── Height: convert "6-8" → inches (e.g. 80.0) ──────────────
        raw_ht = str(row.get('HEIGHT', '') or '')
        if '-' in raw_ht:
            ft, inch = raw_ht.split('-')
            height_in = int(ft) * 12 + int(inch)
        else:
            height_in = None

        player_meta.append({
            'PLAYER_ID':  pid,
            'BIRTH_DATE': row.get('BIRTHDATE',  None),
            'POSITION':   row.get('POSITION',   ''),
            'HEIGHT_IN':  height_in,
            'DRAFT_YEAR': row.get('DRAFT_YEAR', None),
            'COUNTRY':    row.get('COUNTRY',    ''),
        })
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(active_ids)} done...')
        time.sleep(0.65)

    except Exception as e:
        errors_meta.append(pid)
        time.sleep(0.65)

df_meta = pd.DataFrame(player_meta)
df_meta['BIRTH_DATE'] = pd.to_datetime(df_meta['BIRTH_DATE'], errors='coerce')
df_meta['AGE']        = ((DRAFT_DATE - df_meta['BIRTH_DATE']).dt.days / 365.25).round(1)

# ── Height tier (useful for position validation + FG%/REB context) ───────────
def height_tier(h):
    if h is None or pd.isna(h): return 'Unknown'
    if h >= 84: return 'Big (≥7ft)'
    if h >= 80: return 'PF/C (6-8 to 6-11)'
    if h >= 76: return 'Wing (6-4 to 6-7)'
    return 'Guard (<6-4)'

df_meta['HEIGHT_TIER'] = df_meta['HEIGHT_IN'].apply(height_tier)

# ── Normalise POSITION → fantasy slot ────────────────────────────────────────
def normalize_position(pos_str):
    if not isinstance(pos_str, str) or not pos_str.strip():
        return 'G'
    p = pos_str.strip().lower()
    if p in ('center', 'c'):              return 'C'
    if 'center' in p and 'forward' in p:  return 'PF'
    if 'forward-guard' in p:              return 'SF'
    if 'guard-forward' in p:              return 'SG'
    if 'forward' in p:                    return 'PF'
    if 'guard'   in p:                    return 'SG'
    return 'G'

df_meta['POS_PRIMARY'] = df_meta['POSITION'].apply(normalize_position)

print(f'\n✅ Metadata done.  Errors: {len(errors_meta)} skipped.')
print(df_meta[['PLAYER_ID','POSITION','POS_PRIMARY','HEIGHT_IN','HEIGHT_TIER','AGE']].head(8))

Fetching metadata for 572 active players...
Estimated time: ~6 min
  50/572 done...
  100/572 done...
  150/572 done...
  200/572 done...
  250/572 done...
  300/572 done...
  350/572 done...
  400/572 done...
  450/572 done...
  500/572 done...
  550/572 done...

✅ Metadata done.  Errors: 0 skipped.
   PLAYER_ID        POSITION POS_PRIMARY  HEIGHT_IN         HEIGHT_TIER   AGE
0    1642964           Guard          SG         77   Wing (6-4 to 6-7)  24.6
1    1642959           Guard          SG         76   Wing (6-4 to 6-7)  24.6
2    1641717           Guard          SG         75        Guard (<6-4)  22.9
3    1631119         Forward          PF         81  PF/C (6-8 to 6-11)  24.3
4    1631096  Center-Forward          PF         85          Big (≥7ft)  24.4
5    1630598           Guard          SG         77   Wing (6-4 to 6-7)  27.7
6    1629652           Guard          SG         76   Wing (6-4 to 6-7)  27.5
7    1628983           Guard          SG         78   Wing (6-4 to 6-7)  2

In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — CLEAN + FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════

def parse_minutes(val):
    if isinstance(val, str) and ':' in val:
        ft, sc = val.split(':')
        return int(ft) + int(sc) / 60
    try:
        return float(val) if pd.notna(val) else 0.0
    except:
        return 0.0

def clean_and_prep(df):
    df = df.copy()
    df['MIN']       = df['MIN'].apply(parse_minutes)
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    dd_cats = ['PTS','REB','AST','STL','BLK']
    df['DD']     = ((df[dd_cats] >= 10).sum(axis=1) >= 2).astype(int)
    df['STOCKS'] = df['STL'] + df['BLK']
    return df

df_clean = clean_and_prep(df_all_stats)
print(f'✅ Cleaned. Shape: {df_clean.shape}')

✅ Cleaned. Shape: (50233, 35)


In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — AGGREGATE STATS  (rookie-aware)
# ═══════════════════════════════════════════════════════════════════
# Standard:  GP ≥ 15,  MPG ≥ 12
# Rookie:    GP ≥  5,  MPG ≥ 10  → small-sample valid with later shrinkage

def aggregate_stats(df, label='', is_rookie_pass=False):
    agg = {
        'GAME_ID':'count','MIN':'sum','FGM':'sum','FGA':'sum',
        'FTM':'sum','FTA':'sum','FG3M':'sum','REB':'sum',
        'AST':'sum','STL':'sum','BLK':'sum','PTS':'sum','DD':'sum'
    }
    g = df.groupby(['PLAYER_ID','PLAYER_NAME']).agg(agg).reset_index()
    g.rename(columns={'GAME_ID':'GP'}, inplace=True)
    g['MPG'] = g['MIN'] / g['GP']

    if is_rookie_pass:
        g = g[(g['GP'] >= 5) & (g['MPG'] >= 10)]
        g['IS_ROOKIE_SAMPLE'] = True
    else:
        g = g[(g['GP'] >= 15) & (g['MPG'] >= 12)]
        g['IS_ROOKIE_SAMPLE'] = False

    for s in ['FG3M','REB','AST','STL','BLK','PTS','DD']:
        g[f'{s}_PG'] = g[s] / g['GP']

    g['FG_PCT']  = np.where(g['FGA'] > 0, g['FGM'] / g['FGA'],  np.nan)
    g['FT_PCT']  = np.where(g['FTA'] > 0, g['FTM'] / g['FTA'],  np.nan)
    g['PTS_MIN'] = np.where(g['MIN'] > 0, g['PTS'] / g['MIN'],  np.nan)

    if label:
        g = g.add_suffix(f'_{label}')
        g.rename(columns={f'PLAYER_ID_{label}':'PLAYER_ID',
                           f'PLAYER_NAME_{label}':'PLAYER_NAME'}, inplace=True)
    return g


# Identify 2025-26 rookies from metadata
rookie_ids = set()
if len(df_meta):
    dy = pd.to_numeric(df_meta['DRAFT_YEAR'], errors='coerce')
    rookie_ids = set(df_meta[dy == DRAFT_YEAR_CURRENT]['PLAYER_ID'].tolist())

df_current = df_clean[df_clean['SEASON'] == CURRENT_SEASON]
df_prev    = df_clean[df_clean['SEASON'] == PREV_SEASON]

# 2025-26 stats
df_26_std     = aggregate_stats(df_current[~df_current['PLAYER_ID'].isin(rookie_ids)], label='26')
df_26_rook    = aggregate_stats(df_current[ df_current['PLAYER_ID'].isin(rookie_ids)], label='26', is_rookie_pass=True)
df_26         = pd.concat([df_26_std, df_26_rook], ignore_index=True)

# 2024-25 stats
df_25         = aggregate_stats(df_prev, label='25')

# Late-season hot streak (last 15 games of current season)
last_15 = df_current.sort_values('GAME_DATE').groupby('PLAYER_ID').tail(15)
df_late = aggregate_stats(last_15, label='LATE')

print(f'✅ 2025-26 pool : {len(df_26)} players  (rookies: {len(df_26_rook)})')
print(f'✅ 2024-25 pool : {len(df_25)} players')

✅ 2025-26 pool : 393 players  (rookies: 39)
✅ 2024-25 pool : 394 players


In [6]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — RELIABILITY + AGE FACTOR
# ═══════════════════════════════════════════════════════════════════
# Reliability = average GP across both available seasons.
# For rookies who only have 2025-26: reliability capped at 0.65 (uncertainty).

gp_history = (
    df_clean.groupby(['PLAYER_ID','PLAYER_NAME','SEASON'])['GAME_ID']
    .count().reset_index()
)
reliability = (
    gp_history.groupby(['PLAYER_ID','PLAYER_NAME'])['GAME_ID']
    .mean().reset_index()
    .rename(columns={'GAME_ID':'AVG_GP'})
)
reliability['RELIABILITY_SCORE'] = (reliability['AVG_GP'] / MAX_GAMES).clip(0.20, 1.0)

# Age factor: peak = 23-28, growth below, decay above
def age_factor(age):
    if pd.isna(age): return 1.0
    if age < 21:     return 1.14   # top rookie/soph breakout
    if age < 23:     return 1.09
    if age < 25:     return 1.04
    if age <= 28:    return 1.00   # prime
    if age <= 30:    return 0.97
    if age <= 32:    return 0.92
    if age <= 34:    return 0.86
    return 0.80

df_meta['AGE_FACTOR'] = df_meta['AGE'].apply(age_factor)

print('✅ Reliability + age done.')
print(df_meta[['PLAYER_ID','POS_PRIMARY','HEIGHT_IN','AGE','AGE_FACTOR']].head(10))

✅ Reliability + age done.
   PLAYER_ID POS_PRIMARY  HEIGHT_IN   AGE  AGE_FACTOR
0    1642964          SG         77  24.6        1.04
1    1642959          SG         76  24.6        1.04
2    1641717          SG         75  22.9        1.09
3    1631119          PF         81  24.3        1.04
4    1631096          PF         85  24.4        1.04
5    1630598          SG         77  27.7        1.00
6    1629652          SG         76  27.5        1.00
7    1628983          SG         78  28.2        0.97
8    1627936          SG         77  32.6        0.86
9    1642263          SG         74  22.3        1.09


In [7]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — MASTER MERGE + PREDICTIONS
# ═══════════════════════════════════════════════════════════════════

ALL_CATS = ['PTS_PG','AST_PG','FG3M_PG','REB_PG','STL_PG',
            'BLK_PG','FG_PCT','FT_PCT','DD_PG','PTS_MIN']

# 6 correlated target categories — punt the rest
PUNT_CATS   = ['REB_PG','BLK_PG','DD_PG','FG_PCT']   # we concede these
TARGET_CATS = ['PTS_PG','AST_PG','STL_PG','PTS_MIN','FG3M_PG','FT_PCT']

df_master = (
    df_26
    .merge(df_25,                                                   on=['PLAYER_ID','PLAYER_NAME'], how='left')
    .merge(reliability[['PLAYER_ID','RELIABILITY_SCORE']],          on='PLAYER_ID',                how='left')
    .merge(df_late[['PLAYER_ID','PTS_PG_LATE','PTS_MIN_LATE',
                    'AST_PG_LATE','STL_PG_LATE','FG3M_PG_LATE']],   on='PLAYER_ID',                how='left')
    .merge(df_meta[['PLAYER_ID','POS_PRIMARY','AGE','AGE_FACTOR',
                    'HEIGHT_IN','HEIGHT_TIER','DRAFT_YEAR']],         on='PLAYER_ID',                how='left')
)

df_master['RELIABILITY_SCORE'] = df_master['RELIABILITY_SCORE'].fillna(0.60)
df_master['AGE_FACTOR']        = df_master['AGE_FACTOR'].fillna(1.0)
df_master['POS_PRIMARY']       = df_master['POS_PRIMARY'].fillna('G')
df_master['IS_ROOKIE']         = df_master['PLAYER_ID'].isin(rookie_ids)
df_master['IS_ROOKIE_SAMPLE']  = df_master['IS_ROOKIE_SAMPLE_26'].fillna(False)

# ── Weighted Predictions ──────────────────────────────────────────────────────
for cat in ALL_CATS:
    c26, c25, pred = f'{cat}_26', f'{cat}_25', f'PRED_{cat}'
    has_prior = df_master[c25].notna()

    df_master[pred] = np.where(
        has_prior,
        df_master[c26] * 0.70 + df_master[c25] * 0.30,
        df_master[c26]   # rookie or new player: only current season
    ) * df_master['AGE_FACTOR']

    # Shrink rookies 15% toward league mean (small-sample correction)
    mu = df_master[pred].mean()
    df_master.loc[df_master['IS_ROOKIE_SAMPLE'], pred] = (
        df_master.loc[df_master['IS_ROOKIE_SAMPLE'], pred] * 0.85 + mu * 0.15
    )

# ── Late-season breakout boost (target cats only) ─────────────────────────────
for surge in ['PTS_PG','PTS_MIN','AST_PG','STL_PG','FG3M_PG']:
    late_col, pred_col = f'{surge}_LATE', f'PRED_{surge}'
    if late_col not in df_master.columns: continue
    mask = (
        df_master[late_col].notna() &
        (df_master[late_col] > df_master[f'{surge}_26'] * 1.15)
    )
    df_master.loc[mask, pred_col] *= 1.08

# ── Refine PG: top-25 assist guards ──────────────────────────────────────────
guard_mask = df_master['POS_PRIMARY'].isin(['G','SG'])
top_pg_idx = df_master[guard_mask].nlargest(25, 'PRED_AST_PG').index
df_master.loc[top_pg_idx, 'POS_PRIMARY'] = 'PG'

print(f'✅ Master: {len(df_master)} players  (rookies: {df_master["IS_ROOKIE"].sum()})')

✅ Master: 393 players  (rookies: 39)


In [8]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — Z-SCORES:  Full 10-cat  +  6-cat Punt
# ═══════════════════════════════════════════════════════════════════
# Two composite scores are computed side by side:
#   DRAFT_VALUE_TOTAL  →  balanced across all 10 categories
#   PUNT_VALUE         →  maximises only the 6 target categories
#                         (PTS · AST · STL · PTS_MIN · FG3M · FT%)
#
# The 6 cats are correlated because they're dominated by high-usage
# guards and wings. Punting REB/BLK/DD/FG% lets you stack these
# players without the positional penalty of needing bigs.

df_master['EXP_GP'] = (df_master['RELIABILITY_SCORE'] * MAX_GAMES).round(0)

df_master['RISK_FLAG'] = np.select(
    [
        df_master['RELIABILITY_SCORE'] >= 0.85,
        df_master['RELIABILITY_SCORE'] >= 0.70,
        df_master['IS_ROOKIE'],
    ],
    ['ELITE DURABILITY','STABLE','ROOKIE RISK'],
    default='HIGH RISK'
)

# Z-scores for every predicted cat
for cat in ALL_CATS:
    col = f'PRED_{cat}'
    df_master[f'Z_{cat}'] = (df_master[col] - df_master[col].mean()) / df_master[col].std()

# Volume-adjusted Z
gp_ratio = df_master['EXP_GP'] / MAX_GAMES
for cat in ALL_CATS:
    df_master[f'V_{cat}'] = df_master[f'Z_{cat}'] * gp_ratio

# ── Full 10-cat composite ─────────────────────────────────────────────────────
df_master['DRAFT_VALUE_TOTAL'] = df_master[[f'V_{c}' for c in ALL_CATS]].sum(axis=1)

# ── 6-cat PUNT composite (strategy value) ────────────────────────────────────
# Extra weight on FT% and PTS_MIN because they most differentiate elite guards
PUNT_WEIGHTS = {
    'V_PTS_PG':   1.2,
    'V_AST_PG':   1.1,
    'V_STL_PG':   1.3,   # rare and hard to find → premium
    'V_PTS_MIN':  1.2,   # efficiency marker
    'V_FG3M_PG':  1.0,
    'V_FT_PCT':   1.1,
}

df_master['PUNT_VALUE'] = sum(
    df_master[col] * w for col, w in PUNT_WEIGHTS.items()
)

# Offensive/defensive axes for visualisation
df_master['OFFENSE_AXIS'] = df_master[['Z_PTS_PG','Z_AST_PG','Z_PTS_MIN','Z_FG3M_PG','Z_FT_PCT','Z_STL_PG']].sum(axis=1)
df_master['DEFENSE_AXIS'] = df_master[['Z_REB_PG','Z_FG_PCT','Z_BLK_PG','Z_DD_PG']].sum(axis=1)

top15 = df_master.nlargest(15,'PUNT_VALUE')[['PLAYER_NAME','POS_PRIMARY','AGE',
                                               'HEIGHT_IN','PUNT_VALUE','DRAFT_VALUE_TOTAL']]
print('\n─── TOP 15 by PUNT VALUE (6-cat strategy) ───')
print(top15.to_string(index=False))


─── TOP 15 by PUNT VALUE (6-cat strategy) ───
            PLAYER_NAME POS_PRIMARY  AGE  HEIGHT_IN  PUNT_VALUE  DRAFT_VALUE_TOTAL
            Luka Dončić          SF 27.6         80   13.168152          14.902364
Shai Gilgeous-Alexander          PG 28.2         78   13.009056          12.734134
           Tyrese Maxey          PG 25.9         74   11.328067          10.189687
        Anthony Edwards          SG 25.2         76   11.139947          10.699731
        Cade Cunningham          PG 25.0         78   10.904379          13.685838
           Nikola Jokić           C 31.6         83   10.088533          16.646531
       Donovan Mitchell          SG 30.1         74    9.438384           7.305970
            LaMelo Ball          PG 25.1         79    9.075237           7.926631
         Keyonte George          PG 22.9         76    9.037346           7.588621
           Jamal Murray          PG 29.6         76    8.811423           8.288527
      Victor Wembanyama          PF 22.7

In [9]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — AUCTION VALUES  (dual: balanced + 6-cat punt)
# ═══════════════════════════════════════════════════════════════════

NUM_TEAMS       = 10
BUDGET_PER_TEAM = 200
TOTAL_BUDGET    = NUM_TEAMS * BUDGET_PER_TEAM   # $2 000
ROSTER_SIZE     = 13
TOTAL_DRAFTED   = NUM_TEAMS * ROSTER_SIZE        # 130
MIN_BID         = 1

def compute_auction_values(df, value_col, suffix):
    """Attach EST_$ to df based on Value Above Replacement for value_col."""
    df = df.sort_values(value_col, ascending=False).reset_index(drop=True)
    repl = df.iloc[TOTAL_DRAFTED - 1][value_col]
    df[f'VAR_{suffix}'] = (df[value_col] - repl).clip(lower=0)

    remaining = TOTAL_BUDGET - TOTAL_DRAFTED * MIN_BID
    tot_var   = df[f'VAR_{suffix}'].head(TOTAL_DRAFTED).sum()

    df[f'EST_${suffix}'] = (
        (df[f'VAR_{suffix}'] / tot_var) * remaining + MIN_BID
    ).round(0).astype(int)
    df.loc[TOTAL_DRAFTED:, f'EST_${suffix}'] = 0
    return df

df_auction = compute_auction_values(df_master.copy(), 'DRAFT_VALUE_TOTAL', 'BAL')   # balanced
df_auction = compute_auction_values(df_auction,        'PUNT_VALUE',        'PUNT')  # 6-cat punt

SHOW_COLS = [
    'PLAYER_NAME','POS_PRIMARY','AGE','HEIGHT_IN','HEIGHT_TIER','IS_ROOKIE',
    'EST_$PUNT','EST_$BAL','EXP_GP','RISK_FLAG',
    'PRED_PTS_PG','PRED_AST_PG','PRED_STL_PG','PRED_FG3M_PG',
    'PRED_FT_PCT','PRED_PTS_MIN',
    'PRED_REB_PG','PRED_BLK_PG','PRED_DD_PG','PRED_FG_PCT',
    'PUNT_VALUE','DRAFT_VALUE_TOTAL'
]

# Sort by PUNT_VALUE for display
df_auction = df_auction.sort_values('PUNT_VALUE', ascending=False).reset_index(drop=True)

print(f"\n{'─'*80}")
print(f"  2026-27 AUCTION DRAFT BOARD  |  ${BUDGET_PER_TEAM} budget  |  6-CAT PUNT STRATEGY")
print(f"{'─'*80}")
print(df_auction[SHOW_COLS].head(40).to_string(index=False))


────────────────────────────────────────────────────────────────────────────────
  2026-27 AUCTION DRAFT BOARD  |  $200 budget  |  6-CAT PUNT STRATEGY
────────────────────────────────────────────────────────────────────────────────
             PLAYER_NAME POS_PRIMARY  AGE  HEIGHT_IN        HEIGHT_TIER  IS_ROOKIE  EST_$PUNT  EST_$BAL  EXP_GP        RISK_FLAG  PRED_PTS_PG  PRED_AST_PG  PRED_STL_PG  PRED_FG3M_PG  PRED_FT_PCT  PRED_PTS_MIN  PRED_REB_PG  PRED_BLK_PG  PRED_DD_PG  PRED_FG_PCT  PUNT_VALUE  DRAFT_VALUE_TOTAL
             Luka Dončić          SF 27.6         80 PF/C (6-8 to 6-11)      False         58        57    56.0        HIGH RISK    32.033484     8.067355     1.820462      4.158766     0.777768      0.894044     7.941097     0.509871    0.498581     0.468424   13.168152          14.902364
 Shai Gilgeous-Alexander          PG 28.2         78  Wing (6-4 to 6-7)      False         57        48    69.0           STABLE    30.833896     6.835457     1.582561      1.774038    

In [10]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 — OPTIMAL TEAM BUILDER  (6-cat punt strategy)
# ═══════════════════════════════════════════════════════════════════
# Positional requirements (10 starters):
#   PG · SG · SF · PF · C · G-UTIL · F-UTIL · F2-UTIL · G2-UTIL · FLEX
# + 3 bench slots
#
# 6-cat punt twist:
#   • Sort by PUNT_VALUE, not DRAFT_VALUE_TOTAL
#   • Prefer guards & wings in every flex/util slot
#   • Only draft a true C or PF if no other eligible player can fill the slot
#   • Budget split: 88% starters / 12% bench (we want best 10 players)

STARTER_SLOTS = [
    ('PG',   ['PG']),
    ('SG',   ['SG','G']),
    ('SF',   ['SF','F']),
    ('PF',   ['PF','F']),
    ('C',    ['C']),
    ('G',    ['PG','SG','G']),
    ('F',    ['SF','PF','F']),
    ('F2',   ['SF','PF','F']),
    ('G2',   ['PG','SG','G']),
    ('FLEX', ['PG','SG','G','SF','F','PF','C']),
]
BENCH_COUNT = 3


def build_optimal_team(budget=200, pool=None, exclude_ids=None,
                       value_col='PUNT_VALUE', dollar_col='EST_$PUNT',
                       starter_pct=0.88):
    """
    Greedy best-value team within positional and budget constraints.

    Parameters
    ----------
    budget      : total dollars available
    pool        : player DataFrame to pick from (defaults to df_auction)
    exclude_ids : set of PLAYER_IDs already taken
    value_col   : column to sort players by ('PUNT_VALUE' or 'DRAFT_VALUE_TOTAL')
    dollar_col  : column holding estimated auction price
    starter_pct : fraction of budget reserved for 10 starters
    """
    if pool is None:
        pool = df_auction[df_auction[dollar_col] > 0].copy()
    if exclude_ids:
        pool = pool[~pool['PLAYER_ID'].isin(exclude_ids)].copy()

    STARTER_BUDGET = int(budget * starter_pct)
    BENCH_BUDGET   = budget - STARTER_BUDGET
    used_ids = set()
    roster   = []
    remaining = STARTER_BUDGET

    for slot_name, eligible_pos in STARTER_SLOTS:
        slots_left = len(STARTER_SLOTS) - len(roster)
        # Leave at least $1 per remaining slot
        max_spend  = remaining - (slots_left - 1)

        candidates = pool[
            pool['POS_PRIMARY'].isin(eligible_pos) &
            ~pool['PLAYER_ID'].isin(used_ids) &
            (pool[dollar_col] <= max_spend)
        ].sort_values(value_col, ascending=False)

        if candidates.empty:
            # Fallback: ignore budget floor — just fill the slot
            candidates = pool[
                pool['POS_PRIMARY'].isin(eligible_pos) &
                ~pool['PLAYER_ID'].isin(used_ids) &
                (pool[dollar_col] <= remaining)
            ].sort_values(value_col, ascending=False)

        if candidates.empty:
            print(f'  ⚠  Could not fill slot {slot_name}')
            continue

        pick = candidates.iloc[0]
        roster.append({**pick.to_dict(), 'SLOT': slot_name, 'ROLE': 'STARTER'})
        used_ids.add(pick['PLAYER_ID'])
        remaining -= int(pick[dollar_col])

    # ── Bench: best remaining PUNT_VALUE within leftover budget ───────────────
    remaining += BENCH_BUDGET  # add bench budget back in
    for i in range(BENCH_COUNT):
        bench_cands = pool[
            ~pool['PLAYER_ID'].isin(used_ids) &
            (pool[dollar_col] <= remaining)
        ].sort_values(value_col, ascending=False)

        if bench_cands.empty:
            break
        pick = bench_cands.iloc[0]
        roster.append({**pick.to_dict(), 'SLOT': f'BENCH_{i+1}', 'ROLE': 'BENCH'})
        used_ids.add(pick['PLAYER_ID'])
        remaining -= int(pick[dollar_col])

    return pd.DataFrame(roster), budget - remaining


# ── Build YOUR team ───────────────────────────────────────────────────────────
my_team, spent = build_optimal_team(budget=200)

DISPLAY_COLS = [
    'SLOT','ROLE','PLAYER_NAME','POS_PRIMARY','AGE','HEIGHT_IN','IS_ROOKIE',
    'EST_$PUNT','EXP_GP','RISK_FLAG',
    'PRED_PTS_PG','PRED_AST_PG','PRED_STL_PG','PRED_FG3M_PG',
    'PRED_FT_PCT','PRED_PTS_MIN','PUNT_VALUE'
]

print(f"\n{'═'*85}")
print(f"  🏆  YOUR OPTIMAL TEAM  —  6-CAT PUNT STRATEGY  |  2026-27")
print(f"{'═'*85}")
print(my_team[DISPLAY_COLS].to_string(index=False))
print(f"\n  Budget spent : ${spent} / $200")
print(f"  Punt Value   : {my_team['PUNT_VALUE'].sum():.2f}")
print(f"  Exp. cats won / matchup: PTS · AST · STL · PTS/MIN · 3PM · FT%")
print(f"  Cats conceded          : REB · BLK · DD · FG%")

# ── Punt score per category ───────────────────────────────────────────────────
print(f"\n  ── YOUR TEAM CATEGORY AVERAGES ──")
for cat, label in [
    ('PRED_PTS_PG','PTS/G'), ('PRED_AST_PG','AST/G'),
    ('PRED_STL_PG','STL/G'), ('PRED_PTS_MIN','PTS/MIN'),
    ('PRED_FG3M_PG','3PM/G'),('PRED_FT_PCT','FT%'),
    ('PRED_REB_PG','REB/G'), ('PRED_BLK_PG','BLK/G'),
]:
    avg = my_team[cat].mean()
    tag = '✓ TARGET' if cat.replace('PRED_','').replace('_PG','_PG') in\
        ['PTS_PG','AST_PG','STL_PG','FG3M_PG','FT_PCT','PTS_MIN'] else '  punt'
    print(f"    {label:<10} {avg:>6.3f}  {tag}")


═════════════════════════════════════════════════════════════════════════════════════
  🏆  YOUR OPTIMAL TEAM  —  6-CAT PUNT STRATEGY  |  2026-27
═════════════════════════════════════════════════════════════════════════════════════
   SLOT    ROLE             PLAYER_NAME POS_PRIMARY  AGE  HEIGHT_IN  IS_ROOKIE  EST_$PUNT  EXP_GP   RISK_FLAG  PRED_PTS_PG  PRED_AST_PG  PRED_STL_PG  PRED_FG3M_PG  PRED_FT_PCT  PRED_PTS_MIN  PUNT_VALUE
     PG STARTER Shai Gilgeous-Alexander          PG 28.2         78      False         57    69.0      STABLE    30.833896     6.835457     1.582561      1.774038     0.861750      0.916791   13.009056
     SG STARTER         Anthony Edwards          SG 25.2         76      False         48    68.0      STABLE    28.905020     3.970188     1.429008      3.628983     0.808385      0.809352   11.139947
     SF STARTER             Luka Dončić          SF 27.6         80      False         58    56.0   HIGH RISK    32.033484     8.067355     1.820462      4.158766

In [11]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11 — SIMULATE ALL 10 TEAMS
# ═══════════════════════════════════════════════════════════════════
# Your team (team 0) picks first and uses PUNT_VALUE ordering.
# Opponents simulate balanced strategies with ±15% noise to model
# imperfect draft decisions. They are ranked by DRAFT_VALUE_TOTAL.

all_teams    = {}
globally_used = set()

for i in range(NUM_TEAMS):
    if i == 0:
        t, _    = build_optimal_team(budget=200, exclude_ids=globally_used,
                                     value_col='PUNT_VALUE', dollar_col='EST_$PUNT')
        label   = '🏆 YOUR TEAM'
    else:
        opp = df_auction[df_auction['EST_$BAL'] > 0].copy()
        opp['DRAFT_VALUE_TOTAL'] *= np.random.uniform(0.85, 1.15, len(opp))
        opp = opp.sort_values('DRAFT_VALUE_TOTAL', ascending=False).reset_index(drop=True)
        t, _    = build_optimal_team(budget=200, pool=opp,
                                     exclude_ids=globally_used,
                                     value_col='DRAFT_VALUE_TOTAL',
                                     dollar_col='EST_$BAL')
        label   = f'Team {i+1}'

    all_teams[label] = t
    globally_used.update(t['PLAYER_ID'].tolist())

# ── League summary ────────────────────────────────────────────────────────────
summary_rows = []
for name, tdf in all_teams.items():
    r = {'TEAM': name, 'PLAYERS': len(tdf), 'BUDGET_USED': tdf['EST_$PUNT'].sum()}
    for cat in ['PRED_PTS_PG','PRED_AST_PG','PRED_REB_PG',
                'PRED_STL_PG','PRED_BLK_PG','PRED_FG3M_PG','PRED_FT_PCT','PRED_PTS_MIN']:
        r[cat] = round(tdf[cat].mean(), 3)
    r['PUNT_VALUE_SUM'] = round(tdf['PUNT_VALUE'].sum(), 2)
    summary_rows.append(r)

df_league = pd.DataFrame(summary_rows).sort_values('PUNT_VALUE_SUM', ascending=False)

print('\n' + '═'*100)
print('  📊  LEAGUE OVERVIEW — 10-TEAM SIMULATION')
print('═'*100)
print(df_league.to_string(index=False))

  ⚠  Could not fill slot F2
  ⚠  Could not fill slot FLEX
  ⚠  Could not fill slot F2
  ⚠  Could not fill slot G2
  ⚠  Could not fill slot FLEX
  ⚠  Could not fill slot F2
  ⚠  Could not fill slot G2
  ⚠  Could not fill slot FLEX
  ⚠  Could not fill slot SF
  ⚠  Could not fill slot FLEX
  ⚠  Could not fill slot SF
  ⚠  Could not fill slot FLEX
  ⚠  Could not fill slot SF
  ⚠  Could not fill slot SF
  ⚠  Could not fill slot SF
  ⚠  Could not fill slot SF

════════════════════════════════════════════════════════════════════════════════════════════════════
  📊  LEAGUE OVERVIEW — 10-TEAM SIMULATION
════════════════════════════════════════════════════════════════════════════════════════════════════
       TEAM  PLAYERS  BUDGET_USED  PRED_PTS_PG  PRED_AST_PG  PRED_REB_PG  PRED_STL_PG  PRED_BLK_PG  PRED_FG3M_PG  PRED_FT_PCT  PRED_PTS_MIN  PUNT_VALUE_SUM
🏆 YOUR TEAM       11          200       19.670        3.861        5.353        1.117        0.629         1.944        0.808         0.637  

In [12]:
# ═══════════════════════════════════════════════════════════════════
# CELL 12 — CATEGORY WIN PROJECTION
# ═══════════════════════════════════════════════════════════════════
# Shows category-by-category win/loss + projected season W-L record
# for each head-to-head matchup.

CAT_LABELS = {
    'PRED_PTS_PG':  'PTS/G',
    'PRED_AST_PG':  'AST/G',
    'PRED_FG3M_PG': '3PM/G',
    'PRED_STL_PG':  'STL/G',
    'PRED_FT_PCT':  'FT%',
    'PRED_PTS_MIN': 'PTS/MIN',
    'PRED_REB_PG':  'REB/G',
    'PRED_BLK_PG':  'BLK/G',
    'PRED_DD_PG':   'DD/G',
    'PRED_FG_PCT':  'FG%',
}
TARGET_COLS = {'PRED_PTS_PG','PRED_AST_PG','PRED_FG3M_PG',
               'PRED_STL_PG','PRED_FT_PCT','PRED_PTS_MIN'}

my_avgs = {cat: all_teams['🏆 YOUR TEAM'][cat].mean() for cat in CAT_LABELS}

season_wins = season_losses = 0

print('\n' + '═'*90)
print('  📈  CATEGORY WIN PROJECTION  —  YOUR TEAM vs EACH OPPONENT')
print('  ✓ = win  |  ✗ = loss  |  [★] = target category')
print('═'*90)

header = f"{'OPPONENT':<16}"
for lbl in CAT_LABELS.values():
    header += f'{lbl:>9}'
header += f"{'W-L':>7}"
print(header)
print('─'*90)

for name, tdf in all_teams.items():
    if name == '🏆 YOUR TEAM': continue
    w = l = 0
    row = f'{name:<16}'
    for cat, lbl in CAT_LABELS.items():
        diff = my_avgs[cat] - tdf[cat].mean()
        sym  = '✓' if diff > 0 else '✗'
        star = '★' if cat in TARGET_COLS else ' '
        row += f'{sym}{star}      '
        if diff > 0: w += 1
        else:         l += 1
    row += f'{w}-{l}'
    print(row)
    season_wins   += w
    season_losses += l

print('─'*90)
total_matches = season_wins + season_losses
pct = season_wins / total_matches * 100 if total_matches else 0
print(f"  SEASON CATEGORY RECORD:  {season_wins}-{season_losses}  ({pct:.1f}% win rate)")
print(f"  ★ marks your 6 target categories (expected: win most of these every week)")


══════════════════════════════════════════════════════════════════════════════════════════
  📈  CATEGORY WIN PROJECTION  —  YOUR TEAM vs EACH OPPONENT
  ✓ = win  |  ✗ = loss  |  [★] = target category
══════════════════════════════════════════════════════════════════════════════════════════
OPPONENT            PTS/G    AST/G    3PM/G    STL/G      FT%  PTS/MIN    REB/G    BLK/G     DD/G      FG%    W-L
──────────────────────────────────────────────────────────────────────────────────────────
Team 2          ✓★      ✗★      ✓★      ✓★      ✗★      ✓★      ✗       ✗       ✗       ✗       4-6
Team 3          ✓★      ✓★      ✓★      ✗★      ✓★      ✓★      ✗       ✗       ✗       ✗       5-5
Team 4          ✗★      ✗★      ✗★      ✓★      ✓★      ✗★      ✗       ✓       ✗       ✓       4-6
Team 5          ✓★      ✗★      ✗★      ✓★      ✗★      ✓★      ✗       ✓       ✗       ✗       4-6
Team 6          ✗★      ✗★      ✓★      ✓★      ✓★      ✗★      ✗       ✓       ✗       ✓       5-5
Tea

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 13 — SAVE EXCEL
# ═══════════════════════════════════════════════════════════════════

out_file = 'fantasy_draft_2026_27.xlsx'

with pd.ExcelWriter(out_file, engine='openpyxl') as writer:
    # Big Board sorted by punt value
    df_auction[SHOW_COLS].head(150).to_excel(
        writer, sheet_name='Big Board (Punt)', index=False)

    # Big Board sorted by balanced value
    df_auction.sort_values('DRAFT_VALUE_TOTAL', ascending=False)[
        SHOW_COLS].head(150).to_excel(
        writer, sheet_name='Big Board (Balanced)', index=False)

    # Your optimal team
    my_team[DISPLAY_COLS].to_excel(
        writer, sheet_name='YOUR Optimal Team', index=False)

    # League overview
    df_league.to_excel(
        writer, sheet_name='League Overview', index=False)

    # All 10 teams
    for team_name, tdf in all_teams.items():
        sheet = team_name.replace('🏆 ','').replace(' ','_')[:31]
        cols  = [c for c in DISPLAY_COLS if c in tdf.columns]
        tdf[cols].to_excel(writer, sheet_name=sheet, index=False)

    # Player metadata
    df_meta[['PLAYER_ID','POSITION','POS_PRIMARY','AGE','HEIGHT_IN',
             'HEIGHT_TIER','AGE_FACTOR']].to_excel(
        writer, sheet_name='Player Metadata', index=False)

df_meta[['PLAYER_ID','POSITION','POS_PRIMARY','AGE','HEIGHT_IN',
             'HEIGHT_TIER','AGE_FACTOR']]

,PLAYER_ID,POSITION,POS_PRIMARY,AGE,HEIGHT_IN,HEIGHT_TIER,AGE_FACTOR
0,1642964,Guard,SG,24.6,77,Wing (6-4 to 6-7),1.04
1,1642959,Guard,SG,24.6,76,Wing (6-4 to 6-7),1.04
2,1641717,Guard,SG,22.9,75,Guard (<6-4),1.09
3,1631119,Forward,PF,24.3,81,PF/C (6-8 to 6-11),1.04
4,1631096,Center-Forward,PF,24.4,85,Big (≥7ft),1.04
...,...,...,...,...,...,...,...
567,1631113,Guard,SG,24.0,72,Guard (<6-4),1.04
568,1643047,Forward,PF,24.7,83,PF/C (6-8 to 6-11),1.04
569,1628365,Guard,SG,28.3,76,Wing (6-4 to 6-7),0.97
570,1629610,Guard-Forward,SG,29.1,77,Wing (6-4 to 6-7),0.97
